# k03-embed-minilm
Generated from kaggle/jobs. Requires private input data; no outputs or metrics are fabricated.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", 'transformers==4.46.3', 'datasets>=3,<4', 'accelerate>=1,<2', 'sentence-transformers==3.3.1', 'optimum[onnxruntime]==1.23.3', 'onnx>=1.16,<2', 'onnxruntime>=1.20,<2', 'pyvi>=0.1.1', 'sentencepiece>=0.2', 'protobuf', 'pyarrow', 'pandas', 'scikit-learn', 'scipy'], check=True)


In [ ]:
"""Shared source embedded into each generated standalone Kaggle notebook."""
import os
os.environ['TOKENIZERS_PARALLELISM']='false'
import json
import random
import subprocess
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
print({'torch':torch.__version__,'transformers':transformers.__version__,'seed':SEED})
WORK=Path('/kaggle/working');WORK.mkdir(exist_ok=True)
INPUT=Path('/kaggle/input/ecom-intent-data')
START=time.monotonic()


def input_file(name,required=True):
    candidates=sorted(Path('/kaggle/input').rglob(name))
    if not candidates:
        if required:raise FileNotFoundError(f'Upload {name} to the private ecom-intent-data dataset')
        return None
    if len(candidates)>1:
        raise ValueError(f'Ambiguous input {name}: {candidates}. Attach only one data version.')
    return candidates[0]


def save_json(path,data):
    Path(path).write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False))


def time_budget():
    if time.monotonic()-START>5.5*3600:
        raise RuntimeError('5.5 hour budget reached. Download checkpoint, attach it to the dataset, then resume.')


def normalize_text(text):
    import re,unicodedata
    return re.sub(r'\s+',' ',unicodedata.normalize('NFC',text)).strip()


def preprocess(text):return normalize_text(text).lower()


def segment_vi(text):
    from pyvi import ViTokenizer
    return ViTokenizer.tokenize(preprocess(text))


def export_onnx(source,destination,task):
    """GPU export in Kaggle; local work only consumes the exported ONNX."""
    from optimum.onnxruntime import ORTModelForFeatureExtraction,ORTModelForSequenceClassification,ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    cls=ORTModelForFeatureExtraction if task=='feature-extraction' else ORTModelForSequenceClassification
    exported=cls.from_pretrained(str(source),export=True)
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    exported.save_pretrained(destination)
    quantizer=ORTQuantizer.from_pretrained(exported)
    quantizer.quantize(save_dir=destination,quantization_config=AutoQuantizationConfig.avx2(is_static=False,per_channel=False),file_suffix='int8')
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained(str(source)).save_pretrained(destination)


In [ ]:
"""k03: QC or final sentence embeddings, plus fp32/int8 MiniLM ONNX export."""
from sentence_transformers import SentenceTransformer


def run_embedding():
    if not torch.cuda.is_available():raise RuntimeError('Enable Kaggle GPU')
    model=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',device='cuda')
    model.max_seq_length=128
    # QC is run before final datasets exist. Upload qc_pairs.parquet for this stage.
    qc=input_file('qc_pairs.parquet',required=False)
    if qc:
        pairs=pd.read_parquet(qc)
        # Chunk checkpoints allow resumption; slots are excluded from semantic content.
        import re
        clean=lambda t:preprocess(re.sub(r'\{\{[^}]+\}\}',' ',t))
        scores=[]
        for start in range(0,len(pairs),5000):
            time_budget();part=pairs.iloc[start:start+5000]
            a=model.encode([clean(t) for t in part.text_en],batch_size=256,normalize_embeddings=True,convert_to_numpy=True)
            b=model.encode([clean(t) for t in part.candidate_text],batch_size=256,normalize_embeddings=True,convert_to_numpy=True)
            score=(a*b).sum(axis=1);scores.append(score);np.save(WORK/f'qc_cos_{start:06}.npy',score)
        np.save(WORK/'qc_cos.npy',np.concatenate(scores))
    for name in ['train_multi','val_vi','val_en','test_clean_vi','test_noisy_vi','test_real','test_clean_en']:
        path=input_file(name+'.parquet',required=False)
        if path is None:continue
        frame=pd.read_parquet(path)
        output=np.lib.format.open_memmap(WORK/f'emb_{name}.npy',mode='w+',dtype='float16',shape=(len(frame),model.get_sentence_embedding_dimension()))
        for start in range(0,len(frame),5000):
            time_budget()
            output[start:start+5000]=model.encode([preprocess(t) for t in frame.text.iloc[start:start+5000]],batch_size=256,normalize_embeddings=True,convert_to_numpy=True).astype('float16');output.flush()
            save_json(WORK/f'progress_{name}.json',{'completed':min(start+5000,len(frame)),'rows':len(frame)})
        frame[['src_id','text','aug_type']].to_parquet(WORK/f'ids_{name}.parquet',index=False)
    hf=WORK/'minilm_hf';model[0].auto_model.save_pretrained(hf);model.tokenizer.save_pretrained(hf)
    export_onnx(hf,WORK/'minilm_onnx','feature-extraction')


run_embedding()
